# Unidad III: Muestreo e Intervalos de Confianza - Práctica

## Introducción

En este notebook practicaremos:

* Simulación del Teorema Central del Límite
* Construcción de intervalos de confianza para medias y proporciones
* Detección de outliers con IQR y Z-scores
* Análisis de tamaño de muestra
* Aplicaciones a problemas de negocios

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Bibliotecas importadas correctamente")

## Ejercicio 1: Teorema Central del Límite

### Problema
Una población de salarios tiene distribución asimétrica (no normal) con μ = $45,000 y σ = $12,000.

**Preguntas:**
1. Simular la distribución de medias muestrales para n = 5, 15, 30, 50
2. Verificar que E(X̄) ≈ μ
3. Verificar que σ(X̄) ≈ σ/√n

In [0]:
# Parámetros poblacionales
mu_poblacion = 45000
sigma_poblacion = 12000

print("=" * 70)
print("TEOREMA CENTRAL DEL LÍMITE: Distribución de salarios")
print("=" * 70)
print(f"Población: μ = ${mu_poblacion:,}, σ = ${sigma_poblacion:,}\n")

# Crear población asimétrica (lognormal)
np.random.seed(42)
# Ajustar parámetros para obtener media y desv est deseados
sigma_ln = np.sqrt(np.log(1 + (sigma_poblacion/mu_poblacion)**2))
mu_ln = np.log(mu_poblacion) - 0.5 * sigma_ln**2
poblacion = np.random.lognormal(mu_ln, sigma_ln, size=50000)

print(f"Población simulada (distribución lognormal):")
print(f"  Media: ${poblacion.mean():,.2f}")
print(f"  Desv. Est.: ${poblacion.std():,.2f}")
print(f"  Tamaño: {len(poblacion):,}\n")

# Diferentes tamaños de muestra
tamanos = [5, 15, 30, 50]
num_muestras = 1000

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

resultados = []

for idx, n in enumerate(tamanos):
    # Tomar muchas muestras
    medias = []
    for _ in range(num_muestras):
        muestra = np.random.choice(poblacion, size=n)
        medias.append(muestra.mean())
    
    medias = np.array(medias)
    
    # Estadísticas
    media_de_medias = medias.mean()
    std_de_medias = medias.std()
    error_teorico = poblacion.std() / np.sqrt(n)
    
    resultados.append({
        'n': n,
        'Media de X̄': f'${media_de_medias:,.0f}',
        'σ(X̄) observado': f'${std_de_medias:,.0f}',
        'σ/√n teórico': f'${error_teorico:,.0f}'
    })
    
    # Gráfico
    ax = axes[idx]
    ax.hist(medias, bins=30, density=True, color='skyblue', 
            alpha=0.7, edgecolor='black', label='Medias muestrales')
    
    # Curva normal teórica
    x_vals = np.linspace(medias.min(), medias.max(), 100)
    y_vals = stats.norm.pdf(x_vals, loc=poblacion.mean(), scale=error_teorico)
    ax.plot(x_vals, y_vals, 'r-', linewidth=2, label='Normal teórica')
    
    ax.axvline(poblacion.mean(), color='green', linestyle='--', 
               linewidth=2, label=f'μ = ${poblacion.mean():,.0f}')
    
    ax.set_xlabel('Media muestral ($)')
    ax.set_ylabel('Densidad')
    ax.set_title(f'n = {n} (Error Est. = ${error_teorico:,.0f})')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla de resultados
df_resultados = pd.DataFrame(resultados)
print("\nVerificación del TCL:")
print(df_resultados.to_string(index=False))

print("\n" + "=" * 70)
print("CONCLUSIONES:")
print("=" * 70)
print("1. La distribución de X̄ se aproxima a normal a medida que n aumenta")
print("2. E(X̄) ≈ μ (la media de medias ≈ media poblacional)")
print("3. σ(X̄) ≈ σ/√n (el error estándar disminuye con √n)")
print("4. ¡Funciona incluso con población NO NORMAL!")

## Ejercicio 2: Intervalo de Confianza para la Media

### Problema
Una muestra de 40 clientes tiene un gasto promedio de $850 con desviación estándar de $120.

**Preguntas:**
1. Construir IC del 95% para la media poblacional
2. Construir IC del 90% y 99%
3. Interpretar los resultados
4. ¿Qué pasa si aumentamos el tamaño de muestra?

In [0]:
# Datos de la muestra
n = 40
xbar = 850
s = 120

print("=" * 70)
print("INTERVALO DE CONFIANZA PARA LA MEDIA: Gasto de clientes")
print("=" * 70)
print(f"Muestra: n = {n}, x̄ = ${xbar}, s = ${s}\n")

# Calcular IC para diferentes niveles de confianza
niveles = [0.90, 0.95, 0.99]
resultados_ic = []

for confianza in niveles:
    alpha = 1 - confianza
    gl = n - 1  # grados de libertad
    
    # Valor crítico de t
    t_critico = stats.t.ppf(1 - alpha/2, gl)
    
    # Error estándar
    error_estandar = s / np.sqrt(n)
    
    # Margen de error
    margen = t_critico * error_estandar
    
    # Intervalo
    limite_inf = xbar - margen
    limite_sup = xbar + margen
    
    resultados_ic.append({
        'Confianza': f'{confianza*100:.0f}%',
        'α': f'{alpha:.2f}',
        't crítico': f'{t_critico:.3f}',
        'Margen': f'${margen:.2f}',
        'Límite Inf': f'${limite_inf:.2f}',
        'Límite Sup': f'${limite_sup:.2f}',
        'Amplitud': f'${limite_sup - limite_inf:.2f}'
    })
    
    print(f"{int(confianza*100)}% IC para μ:")
    print(f"  Error estándar: ${error_estandar:.2f}")
    print(f"  t crítico (gl={gl}): {t_critico:.3f}")
    print(f"  Margen de error: ${margen:.2f}")
    print(f"  IC: [${limite_inf:.2f}, ${limite_sup:.2f}]")
    print(f"  Interpretación: Estamos {int(confianza*100)}% confiados de que el gasto")
    print(f"  promedio poblacional está entre ${limite_inf:.2f} y ${limite_sup:.2f}\n")

# Tabla comparativa
df_ic = pd.DataFrame(resultados_ic)
print("\nComparación de Intervalos:")
print(df_ic.to_string(index=False))

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Comparación de IC
y_pos = np.arange(len(niveles))
for i, confianza in enumerate(niveles):
    alpha = 1 - confianza
    t_critico = stats.t.ppf(1 - alpha/2, n-1)
    error_estandar = s / np.sqrt(n)
    margen = t_critico * error_estandar
    
    ax1.errorbar(xbar, i, xerr=margen, fmt='o', markersize=8, 
                 capsize=10, capthick=2, label=f'{int(confianza*100)}% IC')

ax1.axvline(xbar, color='red', linestyle='--', linewidth=2, label=f'x̄ = ${xbar}')
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f'{int(c*100)}%' for c in niveles])
ax1.set_xlabel('Gasto ($)')
ax1.set_ylabel('Nivel de Confianza')
ax1.set_title('Comparación de Intervalos de Confianza')
ax1.legend()
ax1.grid(alpha=0.3)

# Gráfico 2: Efecto del tamaño de muestra (95% IC)
confianza = 0.95
alpha = 1 - confianza
tamanos_muestra = np.arange(10, 201, 10)
amplitudes = []

for n_sim in tamanos_muestra:
    t_crit = stats.t.ppf(1 - alpha/2, n_sim - 1)
    error_est = s / np.sqrt(n_sim)
    margen = t_crit * error_est
    amplitud = 2 * margen
    amplitudes.append(amplitud)

ax2.plot(tamanos_muestra, amplitudes, 'b-', linewidth=2)
ax2.axvline(n, color='red', linestyle='--', linewidth=2, label=f'n actual = {n}')
ax2.axhline(2 * t_critico * s / np.sqrt(n), color='red', linestyle='--', 
            linewidth=1, alpha=0.5)
ax2.set_xlabel('Tamaño de muestra (n)')
ax2.set_ylabel('Amplitud del IC ($)')
ax2.set_title('Efecto del tamaño de muestra en la precisión (95% IC)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("OBSERVACIONES:")
print("=" * 70)
print("1. Mayor confianza → Mayor amplitud del intervalo")
print("2. Mayor tamaño de muestra → Menor amplitud (más precisión)")
print("3. La amplitud disminuye proporcionalmente a 1/√n")

## Ejercicio 3: Intervalo de Confianza para una Proporción

### Problema
En una encuesta de 200 clientes, 140 están satisfechos con el servicio.

**Preguntas:**
1. Calcular el IC del 95% para la proporción de clientes satisfechos
2. ¿Podemos afirmar que más del 65% están satisfechos?
3. ¿Qué tamaño de muestra necesitamos para un margen de error de ±3%?

In [0]:
# Datos
n = 200
x = 140  # clientes satisfechos
p_gorro = x / n

print("=" * 70)
print("INTERVALO DE CONFIANZA PARA PROPORCIÓN: Satisfacción de clientes")
print("=" * 70)
print(f"Muestra: n = {n}, x = {x}")
print(f"Proporción muestral: p̂ = {x}/{n} = {p_gorro:.4f} ({p_gorro*100:.2f}%)\n")

# 1. IC del 95%
confianza = 0.95
alpha = 1 - confianza

# Método de Wilson (más preciso)
ci_lower, ci_upper = proportion_confint(x, n, alpha=alpha, method='wilson')

print(f"1. Intervalo de Confianza del 95% (método Wilson):")
print(f"   IC: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"   IC: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"   Amplitud: {(ci_upper - ci_lower)*100:.2f}%")

# Método normal (aproximación)
z_critico = stats.norm.ppf(1 - alpha/2)
error_estandar = np.sqrt(p_gorro * (1 - p_gorro) / n)
margen = z_critico * error_estandar

ci_lower_normal = p_gorro - margen
ci_upper_normal = p_gorro + margen

print(f"\n   Método Normal (aproximación):")
print(f"   Error estándar: {error_estandar:.4f}")
print(f"   z crítico: {z_critico:.3f}")
print(f"   Margen: ±{margen:.4f} (±{margen*100:.2f}%)")
print(f"   IC: [{ci_lower_normal:.4f}, {ci_upper_normal:.4f}]")
print(f"   IC: [{ci_lower_normal*100:.2f}%, {ci_upper_normal*100:.2f}%]")

# 2. ¿Más del 65% satisfechos?
print(f"\n2. ¿Podemos afirmar que más del 65% están satisfechos?")
print(f"   Límite inferior del IC: {ci_lower*100:.2f}%")
if ci_lower > 0.65:
    print(f"   SÍ, con 95% de confianza p > 65%")
else:
    print(f"   NO podemos estar seguros al 95% de confianza")
    print(f"   El IC incluye valores menores a 65%")

# 3. Tamaño de muestra para margen ±3%
margen_deseado = 0.03
p_estimado = p_gorro  # usar proporción observada

n_necesario = (z_critico**2 * p_estimado * (1 - p_estimado)) / margen_deseado**2
n_necesario = int(np.ceil(n_necesario))

print(f"\n3. Tamaño de muestra para margen de error de ±3%:")
print(f"   n necesario = {n_necesario} clientes")
print(f"   n actual = {n}")
if n_necesario > n:
    print(f"   Se necesitan {n_necesario - n} clientes adicionales")
else:
    print(f"   El tamaño actual es suficiente")

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: IC para diferentes niveles de confianza
niveles_conf = [0.90, 0.95, 0.99]
y_pos = np.arange(len(niveles_conf))

for i, conf in enumerate(niveles_conf):
    ci_l, ci_u = proportion_confint(x, n, alpha=1-conf, method='wilson')
    amplitud = ci_u - ci_l
    
    ax1.errorbar(p_gorro, i, xerr=amplitud/2, fmt='o', markersize=8,
                 capsize=10, capthick=2, label=f'{int(conf*100)}% IC')

ax1.axvline(p_gorro, color='red', linestyle='--', linewidth=2, label=f'p̂ = {p_gorro:.3f}')
ax1.axvline(0.65, color='green', linestyle='--', linewidth=1, 
            alpha=0.5, label='p = 0.65')
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f'{int(c*100)}%' for c in niveles_conf])
ax1.set_xlabel('Proporción')
ax1.set_ylabel('Nivel de Confianza')
ax1.set_title('IC para Proporción de Clientes Satisfechos')
ax1.legend()
ax1.grid(alpha=0.3)

# Gráfico 2: Tamaño de muestra vs margen de error
margenes = np.linspace(0.01, 0.10, 50)
tamanos_n = []

for m in margenes:
    n_calc = (z_critico**2 * p_gorro * (1 - p_gorro)) / m**2
    tamanos_n.append(int(np.ceil(n_calc)))

ax2.plot(margenes * 100, tamanos_n, 'b-', linewidth=2)
ax2.axhline(n, color='red', linestyle='--', linewidth=2, label=f'n actual = {n}')
ax2.axvline(3, color='green', linestyle='--', linewidth=1, 
            alpha=0.5, label='Margen deseado = 3%')
ax2.set_xlabel('Margen de error (%)')
ax2.set_ylabel('Tamaño de muestra necesario')
ax2.set_title('Tamaño de Muestra vs Margen de Error (95% IC)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("CONCLUSIONES:")
print("=" * 70)
print(f"• Con 95% de confianza, entre {ci_lower*100:.1f}% y {ci_upper*100:.1f}%")
print(f"  de los clientes están satisfechos")
print(f"• Para mayor precisión, se necesita aumentar el tamaño de muestra")
print(f"• El margen de error disminuye proporcionalmente a 1/√n")

## Ejercicio 4: Detección de Outliers

### Problema
Analizar un conjunto de datos de ventas diarias para identificar valores atípicos.

**Métodos:**
1. Método IQR (Rango Intercuartílico)
2. Método Z-score

In [0]:
# Datos de ventas diarias (últimos 30 días)
np.random.seed(42)
ventas_normales = np.random.normal(5000, 800, size=27)
ventas_outliers = np.array([8500, 9200, 1200])  # agregar outliers
ventas = np.concatenate([ventas_normales, ventas_outliers])
np.random.shuffle(ventas)

print("=" * 70)
print("DETECCIÓN DE OUTLIERS: Ventas diarias")
print("=" * 70)
print(f"\nDatos: {len(ventas)} días de ventas")
print(f"Media: ${ventas.mean():,.2f}")
print(f"Mediana: ${np.median(ventas):,.2f}")
print(f"Desv. Est.: ${ventas.std():,.2f}")
print(f"Mínimo: ${ventas.min():,.2f}")
print(f"Máximo: ${ventas.max():,.2f}\n")

# ============================================================================
# MÉTODO 1: IQR (Interquartile Range)
# ============================================================================
Q1 = np.percentile(ventas, 25)
Q3 = np.percentile(ventas, 75)
IQR = Q3 - Q1

# Límites para outliers
limite_inferior_iqr = Q1 - 1.5 * IQR
limite_superior_iqr = Q3 + 1.5 * IQR

# Identificar outliers
outliers_iqr = ventas[(ventas < limite_inferior_iqr) | (ventas > limite_superior_iqr)]
normales_iqr = ventas[(ventas >= limite_inferior_iqr) & (ventas <= limite_superior_iqr)]

print("1. MÉTODO IQR (Rango Intercuartílico):")
print(f"   Q1 (percentil 25): ${Q1:,.2f}")
print(f"   Q3 (percentil 75): ${Q3:,.2f}")
print(f"   IQR = Q3 - Q1: ${IQR:,.2f}")
print(f"\n   Límites para outliers:")
print(f"   Inferior: Q1 - 1.5×IQR = ${limite_inferior_iqr:,.2f}")
print(f"   Superior: Q3 + 1.5×IQR = ${limite_superior_iqr:,.2f}")
print(f"\n   Outliers detectados: {len(outliers_iqr)}")
if len(outliers_iqr) > 0:
    print(f"   Valores: {', '.join([f'${v:,.0f}' for v in sorted(outliers_iqr)])}")

# ============================================================================
# MÉTODO 2: Z-SCORE
# ============================================================================
media = ventas.mean()
std = ventas.std()
umbrales_z = [2, 3]  # umbrales comunes

print(f"\n2. MÉTODO Z-SCORE:")
print(f"   Media: ${media:,.2f}")
print(f"   Desv. Est.: ${std:,.2f}\n")

for umbral in umbrales_z:
    z_scores = np.abs((ventas - media) / std)
    outliers_z = ventas[z_scores > umbral]
    normales_z = ventas[z_scores <= umbral]
    
    print(f"   Umbral |Z| > {umbral}:")
    print(f"   Outliers detectados: {len(outliers_z)}")
    if len(outliers_z) > 0:
        for v in sorted(outliers_z):
            z = (v - media) / std
            print(f"     ${v:,.0f} (Z = {z:+.2f})")
    print()

# ============================================================================
# VISUALIZACIÓN
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Boxplot con outliers IQR
ax1 = axes[0, 0]
box_parts = ax1.boxplot(ventas, vert=True, patch_artist=True)
box_parts['boxes'][0].set_facecolor('lightblue')
ax1.axhline(limite_inferior_iqr, color='red', linestyle='--', 
            linewidth=1, alpha=0.7, label='Límites IQR')
ax1.axhline(limite_superior_iqr, color='red', linestyle='--', 
            linewidth=1, alpha=0.7)
ax1.scatter([1] * len(outliers_iqr), outliers_iqr, 
            color='red', s=100, zorder=5, label='Outliers IQR')
ax1.set_ylabel('Ventas ($)')
ax1.set_title('Boxplot con Detección de Outliers (Método IQR)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticks([])

# Gráfico 2: Histograma con límites IQR
ax2 = axes[0, 1]
ax2.hist(normales_iqr, bins=15, color='lightblue', alpha=0.7, 
         edgecolor='black', label='Datos normales')
ax2.hist(outliers_iqr, bins=5, color='red', alpha=0.7, 
         edgecolor='black', label='Outliers')
ax2.axvline(limite_inferior_iqr, color='red', linestyle='--', 
            linewidth=2, label='Límites IQR')
ax2.axvline(limite_superior_iqr, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Ventas ($)')
ax2.set_ylabel('Frecuencia')
ax2.set_title('Histograma con Límites IQR')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Gráfico 3: Serie temporal con outliers
ax3 = axes[1, 0]
dias = np.arange(1, len(ventas) + 1)
colores = ['red' if v in outliers_iqr else 'blue' for v in ventas]
ax3.scatter(dias, ventas, c=colores, s=50, alpha=0.7)
ax3.plot(dias, ventas, 'gray', alpha=0.3, linewidth=1)
ax3.axhline(limite_inferior_iqr, color='red', linestyle='--', 
            linewidth=1, alpha=0.5)
ax3.axhline(limite_superior_iqr, color='red', linestyle='--', 
            linewidth=1, alpha=0.5)
ax3.axhline(media, color='green', linestyle='--', 
            linewidth=2, label=f'Media = ${media:,.0f}')
ax3.set_xlabel('Día')
ax3.set_ylabel('Ventas ($)')
ax3.set_title('Serie Temporal con Outliers Marcados')
ax3.legend()
ax3.grid(alpha=0.3)

# Gráfico 4: Z-scores
ax4 = axes[1, 1]
z_scores = (ventas - media) / std
colores_z = ['red' if abs(z) > 2 else 'blue' for z in z_scores]
ax4.scatter(dias, z_scores, c=colores_z, s=50, alpha=0.7)
ax4.axhline(0, color='green', linestyle='-', linewidth=2, label='Media (Z=0)')
ax4.axhline(2, color='orange', linestyle='--', linewidth=1, label='±2σ')
ax4.axhline(-2, color='orange', linestyle='--', linewidth=1)
ax4.axhline(3, color='red', linestyle='--', linewidth=1, label='±3σ')
ax4.axhline(-3, color='red', linestyle='--', linewidth=1)
ax4.set_xlabel('Día')
ax4.set_ylabel('Z-score')
ax4.set_title('Z-scores de las Ventas')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# RESUMEN COMPARATIVO
# ============================================================================
print("=" * 70)
print("RESUMEN COMPARATIVO:")
print("=" * 70)
print(f"\nMétodo IQR (±1.5×IQR):")
print(f"  • Outliers detectados: {len(outliers_iqr)}")
print(f"  • % de datos: {len(outliers_iqr)/len(ventas)*100:.1f}%")
print(f"  • Ventaja: No asume normalidad, robusto")

z_scores_abs = np.abs((ventas - media) / std)
outliers_z2 = ventas[z_scores_abs > 2]
outliers_z3 = ventas[z_scores_abs > 3]

print(f"\nMétodo Z-score (|Z| > 2):")
print(f"  • Outliers detectados: {len(outliers_z2)}")
print(f"  • % de datos: {len(outliers_z2)/len(ventas)*100:.1f}%")
print(f"  • Ventaja: Basado en desviaciones estándar")

print(f"\nMétodo Z-score (|Z| > 3):")
print(f"  • Outliers detectados: {len(outliers_z3)}")
print(f"  • % de datos: {len(outliers_z3)/len(ventas)*100:.1f}%")
print(f"  • Ventaja: Más estricto, outliers extremos")

print("\n" + "=" * 70)
print("RECOMENDACIONES:")
print("=" * 70)
print("1. Usar IQR cuando los datos no son normales")
print("2. Usar Z-score cuando se asume normalidad")
print("3. Investigar la causa de los outliers antes de eliminarlos")
print("4. Los outliers pueden contener información valiosa")

## Resumen y Conclusiones

### Conceptos Clave

**Teorema Central del Límite:**
* La distribución de medias muestrales tiende a ser normal
* E(X̄) = μ (media de medias = media poblacional)
* σ(X̄) = σ/√n (error estándar)
* Funciona incluso con poblaciones no normales

**Intervalos de Confianza:**
* IC para μ (σ desconocida): x̄ ± t_{α/2,n-1} × (s/√n)
* IC para p: usar método de Wilson o aproximación normal
* Mayor confianza → Mayor amplitud
* Mayor n → Menor amplitud (más precisión)

**Detección de Outliers:**
* **Método IQR:** valores fuera de [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
  * No asume normalidad
  * Robusto
* **Método Z-score:** |Z| > 2 o |Z| > 3
  * Asume normalidad
  * Basado en desviaciones estándar

### Herramientas Python

```python
# Teorema Central del Límite
medias = [np.random.choice(poblacion, n).mean() for _ in range(1000)]

# IC para media
from scipy import stats
stats.t.interval(confidence, df, loc=xbar, scale=s/np.sqrt(n))

# IC para proporción
from statsmodels.stats.proportion import proportion_confint
proportion_confint(count, nobs, alpha, method='wilson')

# Outliers IQR
Q1, Q3 = np.percentile(data, [25, 75])
IQR = Q3 - Q1
outliers = data[(data < Q1-1.5*IQR) | (data > Q3+1.5*IQR)]

# Outliers Z-score
z_scores = np.abs((data - data.mean()) / data.std())
outliers = data[z_scores > 2]
```

### Aplicaciones en Negocios

* Estimar parámetros poblacionales a partir de muestras
* Determinar tamaños de muestra adecuados
* Identificar transacciones o comportamientos anómalos
* Construir intervalos de predicción
* Evaluar la confiabilidad de estimaciones